# Etterforskning

Vi har en stor ferdiglaget graf, med titusenvis av noder.  Vi skal bruke Neo4J til å lete etter kriminalitet.

## "Money mule"


In [ ]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)

podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

In [1]:
# Få Kontakt
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Kontakt!")

Kontakt!


In [20]:
# Ikke starte med gamle data
records, summary, keys = driver.execute_query(
    """match (n) detach delete n""")
#
for s in summary.gql_status_objects:
    print(s)
#

note: successful completion - omitted result


In [3]:
# Sjekke APOC
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")

print(f"Server Address: {summary.server.address}")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")



Server Address: 127.0.0.1:7687
Keys:
	version: <Record version='2025.11.2'>
Ressursbruk
	Kjøringen: 1ms
	Å konsumere: 0ms


In [4]:
# Sjekke GDS
records, summary, keys = driver.execute_query(
    """
    CALL gds.version();
    """)

print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	gdsVersion: <Record gdsVersion='2.24.0'>


Vi flytter grafen vi skal arbeide med

In [7]:
%%bash
cp grafer/Komplett.graphml neo4j/import

In [21]:
# Lese inn grafen vi har bygget tidligere
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("Komplett.graphml", {storeNodeIds: true, readLabels: true})""")
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: Komplett.graphml
	source: file
	format: graphml
	nodes: 56725
	relationships: 92351
	properties: 57312
	time: 1251
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 1ms
	Å konsumere: 1255ms


In [ ]:
records, summary, keys = driver.execute_query(
"""MATCH (n)
    WHERE n.Person IS NOT NULL
    SET n:Person
    RETURN count(n)""")
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	count(n): 56620
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 120ms
	Å konsumere: 141ms
